In [1]:
# Import pandas for data manipulation and analysis.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import matplotlib and seaborn for visualizations.
import matplotlib.pyplot as plt
import seaborn as sns

# Display more columns when inspecting DataFrames.
pd.set_option("display.max_columns", None)

# Display monetary values with two decimal places where applicable.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Define the project structure relative to the notebooks folder.
# Notebook 04 is located inside:
# checkout-incentives-ltv/notebooks/
#
# Therefore, the raw data folder is one level above the notebooks folder.

from pathlib import Path

# Get the current notebook working directory.
BASE_DIR = Path.cwd().parent

# Define the raw-data directory.
RAW_DIR = BASE_DIR / "data" / "raw"

# Display the paths so we can verify the project structure.
print("Project directory:", BASE_DIR)
print("Raw data directory:", RAW_DIR)

# Confirm that the raw-data directory exists.
print("\nRaw folder exists:", RAW_DIR.exists())

Project directory: C:\Users\DELL\Desktop\checkout-incentives-ltv
Raw data directory: C:\Users\DELL\Desktop\checkout-incentives-ltv\data\raw

Raw folder exists: True


In [3]:
# Load the Olist datasets required for repeat-purchase prediction.
#
# We load the original raw files rather than relying on variables from
# previous notebooks. This makes Notebook 04 independently reproducible.

customers = pd.read_csv(
    RAW_DIR / "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    RAW_DIR / "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    RAW_DIR / "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    RAW_DIR / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    RAW_DIR / "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    RAW_DIR / "olist_products_dataset.csv"
)

print("Datasets loaded successfully.\n")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Products:", products.shape)

Datasets loaded successfully.

Customers: (99441, 5)
Orders: (99441, 8)
Order items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)
Products: (32951, 9)


In [4]:
# Convert all order lifecycle timestamps to datetime.
#
# Datetime conversion is necessary because several features in this notebook
# will be based on timing, such as:
# - purchase date
# - delivery duration
# - purchase month
# - purchase day of week

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

# Confirm the resulting data types.
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
# Inspect the order dataset before constructing the modeling population.
#
# This helps us verify the available order statuses and understand whether
# any unexpected values are present.

print("Order status distribution:\n")

print(
    orders["order_status"]
    .value_counts(dropna=False)
)

print("\nMissing values:\n")

print(
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date"
        ]
    ]
    .isna()
    .sum()
)

Order status distribution:

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing values:

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    2965
dtype: int64


In [6]:
# Map each customer_id to the corresponding customer_unique_id.
#
# customer_id identifies an order-linked customer record, while
# customer_unique_id represents the actual customer across multiple orders.
#
# Notebook 02 established that customer_unique_id is the correct level for
# customer-level repeat-purchase analysis.

customer_identity = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_state"
    ]
].copy()

# Check whether customer_id is unique in the customer table.
print(
    "Duplicate customer_id records:",
    customer_identity["customer_id"].duplicated().sum()
)

# Display a sample of the mapping.
customer_identity.head()

Duplicate customer_id records: 0


,customer_id,customer_unique_id,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,SP


In [7]:
# Attach customer_unique_id and customer_state to every order.
#
# This allows us to reconstruct the complete purchase history of each
# individual customer.

orders_model = orders.merge(
    customer_identity,
    on="customer_id",
    how="left"
)

# Verify that the merge did not create unexpected row duplication.
print("Orders before merge:", len(orders))
print("Orders after merge:", len(orders_model))

# Check whether customer identity was successfully attached.
print(
    "\nMissing customer_unique_id:",
    orders_model["customer_unique_id"].isna().sum()
)

orders_model.head()

Orders before merge: 99441
Orders after merge: 99441

Missing customer_unique_id: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP


In [8]:
# For repeat-purchase prediction, we use delivered orders as the observable
# purchase history.
#
# This is consistent with the business interpretation used in Notebooks 01–03:
# a completed/delivered transaction represents an actual customer purchase.

orders_delivered = orders_model[
    orders_model["order_status"] == "delivered"
].copy()

print("Delivered orders:", len(orders_delivered))
print(
    "Unique customers:",
    orders_delivered["customer_unique_id"].nunique()
)

orders_delivered.head()

Delivered orders: 96478
Unique customers: 93358


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,SP


In [9]:
# Count the number of delivered orders for every customer.
#
# This allows us to identify:
# - one-time customers: exactly one delivered order
# - repeat customers: two or more delivered orders

customer_order_counts = (
    orders_delivered
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

# Create the customer type based on historical purchase frequency.
customer_order_counts["customer_type"] = np.where(
    customer_order_counts["total_orders"] == 1,
    "One-time",
    "Repeat"
)

print(
    customer_order_counts["customer_type"]
    .value_counts()
)

print("\nTotal customers:", len(customer_order_counts))

customer_type
One-time    90557
Repeat       2801
Name: count, dtype: int64

Total customers: 93358


In [10]:
# Notebook 02 established the following historical customer counts:
#
# One-time customers = 93,099
# Repeat customers   = 2,997
# Total customers    = 96,096
#
# We verify that Notebook 04 reconstructs the same population.

verification = (
    customer_order_counts["customer_type"]
    .value_counts()
)

print("One-time customers:", verification.get("One-time", 0))
print("Repeat customers:", verification.get("Repeat", 0))
print("Total customers:", len(customer_order_counts))

print("\nExpected from Notebook 02:")
print("One-time customers: 93,099")
print("Repeat customers: 2,997")
print("Total customers: 96,096")

One-time customers: 90557
Repeat customers: 2801
Total customers: 93358

Expected from Notebook 02:
One-time customers: 93,099
Repeat customers: 2,997
Total customers: 96,096


In [11]:
# ============================================================
# REBUILD CUSTOMER-LEVEL ORDER DATASET
# ============================================================
#
# Notebook 02 used the broader order population rather than
# restricting customers to delivered orders only.
#
# We therefore rebuild the customer-level dataset from the
# original orders dataframe so that Notebook 04 uses the same
# customer population as Notebook 02.
#
# This prevents population drift between notebooks.
# ============================================================

# Start with the original orders dataframe.
# We exclude only orders that should not contribute to
# customer purchase/LTV analysis.
#
# "canceled" and "unavailable" orders are excluded because
# they do not represent completed purchases.
orders_analysis = orders[
    ~orders["order_status"].isin(["canceled", "unavailable"])
].copy()

# Convert purchase timestamp to datetime so that all
# chronological calculations work correctly.
orders_analysis["order_purchase_timestamp"] = pd.to_datetime(
    orders_analysis["order_purchase_timestamp"]
)

# Merge customer_unique_id and customer_state from the
# customer dataset.
#
# customer_id identifies an individual order-level account,
# while customer_unique_id identifies the actual customer
# across multiple orders.
orders_analysis = orders_analysis.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# Verify that the merge did not lose any orders.
print("Orders in analysis dataset:", len(orders_analysis))

# Verify that every order has a customer_unique_id.
print(
    "Missing customer_unique_id:",
    orders_analysis["customer_unique_id"].isna().sum()
)

# Count unique customers in the same population used for
# downstream customer-level analysis.
customer_order_counts = (
    orders_analysis
    .groupby("customer_unique_id")["order_id"]
    .nunique()
)

print(
    "\nTotal customers:",
    customer_order_counts.index.nunique()
)

print(
    "One-time customers:",
    (customer_order_counts == 1).sum()
)

print(
    "Repeat customers:",
    (customer_order_counts > 1).sum()
)

Orders in analysis dataset: 98207
Missing customer_unique_id: 0

Total customers: 94990
One-time customers: 92102
Repeat customers: 2888


In [13]:
# ============================================================
# RECREATE NOTEBOOK 02 CUSTOMER-LEVEL DATASET
# ============================================================
#
# Notebook 04 is a separate Jupyter notebook, so variables
# such as `customer_features` from Notebook 02/03 are NOT
# available here.
#
# Therefore, we recreate the customer-level dataset directly
# from the original Olist CSV files.
#
# IMPORTANT:
# We use the same broad order population used for the
# customer-level LTV analysis rather than restricting the
# population to delivered orders.
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Load the original Olist datasets
# ------------------------------------------------------------

customers = pd.read_csv(
    "../data/raw/olist_customers_dataset.csv"
)

orders = pd.read_csv(
    "../data/raw/olist_orders_dataset.csv"
)

# ------------------------------------------------------------
# 2. Convert purchase timestamp to datetime
# ------------------------------------------------------------
#
# This is necessary for calculating:
# - first purchase
# - last purchase
# - customer lifetime
# ------------------------------------------------------------

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

# ------------------------------------------------------------
# 3. Merge orders with customer information
# ------------------------------------------------------------
#
# customer_id identifies the account associated with an order.
#
# customer_unique_id identifies the actual customer and is
# therefore the identifier we use for LTV/repeat-purchase
# analysis.
#
# customer_state is retained for later geographic analysis.
# ------------------------------------------------------------

orders_customer = orders.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# ------------------------------------------------------------
# 4. Create customer-level features
# ------------------------------------------------------------
#
# Each row in the resulting dataframe represents one unique
# customer.
#
# total_orders:
#     Number of orders made by the customer.
#
# first_purchase:
#     Date/time of the customer's first order.
#
# last_purchase:
#     Date/time of the customer's latest order.
#
# customer_lifetime_days:
#     Time between first and last purchase.
# ------------------------------------------------------------

customer_base = (
    orders_customer
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        customer_state=("customer_state", "first")
    )
    .reset_index()
)

# Calculate the number of days between the customer's
# first and last purchase.
customer_base["customer_lifetime_days"] = (
    customer_base["last_purchase"]
    - customer_base["first_purchase"]
).dt.total_seconds() / (60 * 60 * 24)

# ------------------------------------------------------------
# 5. Calculate customer LTV
# ------------------------------------------------------------
#
# The order-level dataset does not contain order_value.
# We therefore calculate order value from the payment dataset.
# ------------------------------------------------------------

payments = pd.read_csv(
    "../data/raw/olist_order_payments_dataset.csv"
)

# Calculate the total payment value for each order.
#
# An order can have multiple payment records, so we aggregate
# them by order_id before merging with the orders dataset.
order_values = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_value"})
)

# Add order value to the order/customer dataset.
orders_customer = orders_customer.merge(
    order_values,
    on="order_id",
    how="left"
)

# Replace missing order values with zero.
# This prevents missing values from affecting the LTV sum.
orders_customer["order_value"] = (
    orders_customer["order_value"]
    .fillna(0)
)

# Calculate total customer LTV by summing the value of all
# orders belonging to each customer.
customer_ltv = (
    orders_customer
    .groupby("customer_unique_id")["order_value"]
    .sum()
    .reset_index()
    .rename(columns={"order_value": "total_ltv"})
)

# Merge LTV into the customer-level dataframe.
customer_base = customer_base.merge(
    customer_ltv,
    on="customer_unique_id",
    how="left"
)

# ------------------------------------------------------------
# 6. Identify repeat customers
# ------------------------------------------------------------
#
# A customer with exactly one order is classified as
# "One-time".
#
# A customer with more than one order is classified as
# "Repeat".
# ------------------------------------------------------------

customer_base["is_repeat_customer"] = (
    customer_base["total_orders"] > 1
).astype(int)

customer_base["customer_type"] = (
    customer_base["is_repeat_customer"]
    .map({
        0: "One-time",
        1: "Repeat"
    })
)

# ------------------------------------------------------------
# 7. Verify the resulting population
# ------------------------------------------------------------

print("Customer dataframe shape:", customer_base.shape)

print("\nCustomer type distribution:")
print(
    customer_base["customer_type"]
    .value_counts()
)

print(
    "\nTotal customers:",
    len(customer_base)
)

print(
    "One-time customers:",
    (customer_base["customer_type"] == "One-time").sum()
)

print(
    "Repeat customers:",
    (customer_base["customer_type"] == "Repeat").sum()
)

print("\nColumns:")
print(customer_base.columns.tolist())

# Show the first five customer records.
display(customer_base.head())

Customer dataframe shape: (96096, 9)

Customer type distribution:
customer_type
One-time    93099
Repeat       2997
Name: count, dtype: int64

Total customers: 96096
One-time customers: 93099
Repeat customers: 2997

Columns:
['customer_unique_id', 'total_orders', 'first_purchase', 'last_purchase', 'customer_state', 'customer_lifetime_days', 'total_ltv', 'is_repeat_customer', 'customer_type']


,customer_unique_id,total_orders,first_purchase,last_purchase,customer_state,customer_lifetime_days,total_ltv,is_repeat_customer,customer_type
0,0000366f3b9a7992bf8c76cfdf3221e2,1,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,0.00,141.90,0,One-time
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,0.00,27.19,0,One-time
2,0000f46a3911fa3c0805444483337064,1,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,0.00,86.22,0,One-time
3,0000f6ccb0745a6a4b88665a16c9f078,1,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,0.00,43.62,0,One-time
4,0004aac84e0df4da2b147fca70cf8255,1,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,0.00,196.89,0,One-time


In [14]:
# ============================================================
# CREATE ORDER-LEVEL DATASET FOR CHECKOUT ANALYSIS
# ============================================================
#
# We now have the correct customer population from Notebook 02.
#
# For Notebook 04, we need to analyze customer behavior around
# the FIRST PURCHASE because the business question is:
#
# "Which customers should receive a checkout incentive to
# encourage a repeat purchase?"
#
# Therefore, we need every order along with:
# - customer_unique_id
# - order_id
# - purchase timestamp
# - order value
#
# We will later identify each customer's first order and
# determine whether they eventually became a repeat customer.
# ============================================================

# ------------------------------------------------------------
# 1. Aggregate payment values at order level
# ------------------------------------------------------------
#
# An Olist order can have multiple payment records.
# Therefore, we first sum payment_value for each order.
# ------------------------------------------------------------

order_values = (
    payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_value"})
)

# ------------------------------------------------------------
# 2. Create order-level dataset
# ------------------------------------------------------------
#
# We use the original orders dataframe and attach the
# calculated order value.
# ------------------------------------------------------------

orders_analysis = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ]
].copy()

# Attach customer_unique_id using the customer table.
orders_analysis = orders_analysis.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

# Attach order value.
orders_analysis = orders_analysis.merge(
    order_values,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# 3. Convert timestamp to datetime
# ------------------------------------------------------------

orders_analysis["order_purchase_timestamp"] = pd.to_datetime(
    orders_analysis["order_purchase_timestamp"]
)

# ------------------------------------------------------------
# 4. Handle missing order values
# ------------------------------------------------------------
#
# Orders without a payment record are assigned zero.
# ------------------------------------------------------------

orders_analysis["order_value"] = (
    orders_analysis["order_value"]
    .fillna(0)
)

# ------------------------------------------------------------
# 5. Verify the resulting dataset
# ------------------------------------------------------------

print("Orders in analysis dataset:", len(orders_analysis))

print(
    "Missing customer_unique_id:",
    orders_analysis["customer_unique_id"].isna().sum()
)

print(
    "Missing order_value:",
    orders_analysis["order_value"].isna().sum()
)

print("\nColumns:")
print(orders_analysis.columns.tolist())

# Show the first five rows.
display(orders_analysis.head())

Orders in analysis dataset: 99441
Missing customer_unique_id: 0
Missing order_value: 0

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'customer_unique_id', 'customer_state', 'order_value']


,order_id,customer_id,order_status,order_purchase_timestamp,customer_unique_id,customer_state,order_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,7c396fd4830fd04220f754e42b4e5bff,SP,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,af07308b275d755c9edb36a90c618231,BA,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,3a653a41f6f9fc3d2a113cf8398680e8,GO,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,7c142cf63193a1473d2e66489a9ae977,RN,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,72632f0f9dd73dfee390c9b22eb56dd6,SP,28.62


In [15]:
# ============================================================
# IDENTIFY EACH CUSTOMER'S FIRST ORDER
# ============================================================
#
# The checkout incentive is intended to encourage a customer
# to make a SECOND purchase.
#
# Therefore, we need to identify:
#   1. The customer's first order
#   2. The value of that first order
#   3. Whether the customer eventually became a repeat buyer
#
# We sort all orders chronologically within each customer and
# then take the first order.
# ============================================================

# ------------------------------------------------------------
# 1. Sort orders chronologically for each customer
# ------------------------------------------------------------
#
# customer_unique_id ensures that we track the actual customer
# rather than the individual customer_id assigned to an order.
# ------------------------------------------------------------

orders_sorted = orders_analysis.sort_values(
    [
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_id"
    ]
).copy()

# ------------------------------------------------------------
# 2. Assign purchase number
# ------------------------------------------------------------
#
# cumcount() starts from 0, so we add 1:
#
# 1 = first purchase
# 2 = second purchase
# 3 = third purchase
# ...
# ------------------------------------------------------------

orders_sorted["purchase_number"] = (
    orders_sorted
    .groupby("customer_unique_id")
    .cumcount()
    + 1
)

# ------------------------------------------------------------
# 3. Extract the first order of each customer
# ------------------------------------------------------------

first_orders = orders_sorted[
    orders_sorted["purchase_number"] == 1
].copy()

# ------------------------------------------------------------
# 4. Add customer type from the authoritative Notebook 02
# dataset.
# ------------------------------------------------------------
#
# This ensures that the repeat/one-time classification is
# exactly consistent with Notebook 02.
# ------------------------------------------------------------

first_orders = first_orders.merge(
    customer_base[
        [
            "customer_unique_id",
            "customer_type",
            "total_orders",
            "total_ltv"
        ]
    ],
    on="customer_unique_id",
    how="left",
    suffixes=("", "_customer")
)

# ------------------------------------------------------------
# 5. Verify the result
# ------------------------------------------------------------

print("First-order dataset shape:", first_orders.shape)

print(
    "\nUnique customers:",
    first_orders["customer_unique_id"].nunique()
)

print("\nCustomer type distribution:")
print(
    first_orders["customer_type"]
    .value_counts()
)

print("\nFirst-order value statistics:")
print(
    first_orders["order_value"].describe()
)

# Display the first five records.
display(
    first_orders[
        [
            "customer_unique_id",
            "order_id",
            "order_purchase_timestamp",
            "order_value",
            "customer_type",
            "total_orders",
            "total_ltv"
        ]
    ].head()
)

First-order dataset shape: (96096, 11)

Unique customers: 96096

Customer type distribution:
customer_type
One-time    93099
Repeat       2997
Name: count, dtype: int64

First-order value statistics:
count   96,096.00
mean       161.41
std        223.22
min          0.00
25%         62.01
50%        105.39
75%        177.20
max     13,664.08
Name: order_value, dtype: float64


,customer_unique_id,order_id,order_purchase_timestamp,order_value,customer_type,total_orders,total_ltv
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27,141.90,One-time,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27,27.19,One-time,1,27.19
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03,86.22,One-time,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41,43.62,One-time,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42,196.89,One-time,1,196.89


In [16]:
# ============================================================
# FIRST-ORDER VALUE VS REPEAT-PURCHASE BEHAVIOR
# ============================================================
#
# Business question:
#
# "Does the value of a customer's first purchase provide
# evidence about their likelihood of making a repeat purchase?"
#
# If certain first-order value ranges have substantially higher
# repeat rates, they may be more attractive segments for a
# checkout incentive.
# ============================================================

# ------------------------------------------------------------
# 1. Create first-order value bands
# ------------------------------------------------------------
#
# These bands allow us to compare customers at different
# first-purchase spending levels.
# ------------------------------------------------------------

bins = [
    -float("inf"),
    50,
    100,
    250,
    500,
    1000,
    float("inf")
]

labels = [
    "< $50",
    "$50–$99",
    "$100–$249",
    "$250–$499",
    "$500–$999",
    "$1,000+"
]

first_orders["first_order_value_band"] = pd.cut(
    first_orders["order_value"],
    bins=bins,
    labels=labels,
    right=False
)

# ------------------------------------------------------------
# 2. Calculate repeat behavior for each value band
# ------------------------------------------------------------
#
# repeat_customers:
#     Number of customers who eventually made another purchase.
#
# repeat_rate_pct:
#     Percentage of customers in that band who became repeat
#     customers.
#
# average_first_order_value:
#     Mean first-order value within the band.
#
# median_first_order_value:
#     Median first-order value within the band.
# ------------------------------------------------------------

first_order_value_analysis = (
    first_orders
    .groupby(
        "first_order_value_band",
        observed=True
    )
    .agg(
        customers=("customer_unique_id", "nunique"),

        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),

        average_first_order_value=(
            "order_value",
            "mean"
        ),

        median_first_order_value=(
            "order_value",
            "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 3. Calculate repeat rate
# ------------------------------------------------------------

first_order_value_analysis["repeat_rate_pct"] = (
    first_order_value_analysis["repeat_customers"]
    / first_order_value_analysis["customers"]
    * 100
)

# ------------------------------------------------------------
# 4. Calculate the percentage of the customer population
# ------------------------------------------------------------

first_order_value_analysis["customer_share_pct"] = (
    first_order_value_analysis["customers"]
    / first_order_value_analysis["customers"].sum()
    * 100
)

# ------------------------------------------------------------
# 5. Round numerical values for readability
# ------------------------------------------------------------

first_order_value_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "repeat_rate_pct",
        "customer_share_pct"
    ]
] = first_order_value_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "repeat_rate_pct",
        "customer_share_pct"
    ]
].round(2)

# ------------------------------------------------------------
# 6. Display the final analysis
# ------------------------------------------------------------

display(first_order_value_analysis)

,first_order_value_band,customers,repeat_customers,average_first_order_value,median_first_order_value,repeat_rate_pct,customer_share_pct
0,< $50,16334,554,36.88,37.53,3.39,17.00
1,$50–$99,29247,928,73.29,72.15,3.17,30.44
2,$100–$249,37236,1152,155.72,148.15,3.09,38.75
3,$250–$499,9090,265,337.34,323.04,2.92,9.46
4,$500–$999,3044,75,680.82,652.09,2.46,3.17
5,"$1,000+",1145,23,"1,596.27","1,367.50",2.01,1.19


In [17]:
# ============================================================
# CUSTOMER STATE VS REPEAT-PURCHASE BEHAVIOR
# ============================================================
#
# Business question:
#
# "Does repeat-purchase behavior differ across customer states?"
#
# If some states have consistently higher repeat rates, they may
# represent stronger opportunities for checkout incentives.
#
# IMPORTANT:
# customer_state is already available in first_orders, so we do
# not need to merge another dataset here.
# ============================================================


# ------------------------------------------------------------
# 1. Group customers by state
# ------------------------------------------------------------
#
# Each row in first_orders represents one customer's first order.
# Therefore, grouping here gives us customer-level repeat rates
# without double-counting customers.
# ------------------------------------------------------------

state_repeat_analysis = (
    first_orders
    .groupby(
        "customer_state",
        observed=True
    )
    .agg(
        # Number of customers whose first order was in the state.
        customers=(
            "customer_unique_id",
            "nunique"
        ),

        # Number of those customers who eventually became repeat
        # purchasers.
        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),

        # Average value of the customer's first order.
        average_first_order_value=(
            "order_value",
            "mean"
        ),

        # Median value of the customer's first order.
        median_first_order_value=(
            "order_value",
            "median"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. Calculate repeat rate
# ------------------------------------------------------------
#
# Repeat rate tells us what percentage of customers in each
# state eventually made another purchase.
# ------------------------------------------------------------

state_repeat_analysis["repeat_rate_pct"] = (
    state_repeat_analysis["repeat_customers"]
    / state_repeat_analysis["customers"]
    * 100
)


# ------------------------------------------------------------
# 3. Sort states by repeat rate
# ------------------------------------------------------------
#
# Highest-repeat states appear first.
# ------------------------------------------------------------

state_repeat_analysis = (
    state_repeat_analysis
    .sort_values(
        "repeat_rate_pct",
        ascending=False
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Round monetary and percentage values
# ------------------------------------------------------------

state_repeat_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "repeat_rate_pct"
    ]
] = state_repeat_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "repeat_rate_pct"
    ]
].round(2)


# ------------------------------------------------------------
# 5. Display the complete state-level analysis
# ------------------------------------------------------------

display(state_repeat_analysis)

,customer_state,customers,repeat_customers,average_first_order_value,median_first_order_value,repeat_rate_pct
0,AC,77,4,244.02,158.27,5.19
1,RO,239,10,246.27,157.68,4.18
2,MT,875,30,208.13,125.90,3.43
3,RJ,12379,424,167.30,112.42,3.43
4,GO,1951,65,174.44,113.44,3.33
5,AL,401,13,234.70,138.15,3.24
6,SP,40291,1302,143.92,93.85,3.23
7,RS,5276,168,163.03,108.54,3.18
8,DF,2074,64,167.10,108.20,3.09
9,MG,11256,345,161.49,108.63,3.07


In [18]:
# ============================================================
# DELIVERY EXPERIENCE VS REPEAT-PURCHASE BEHAVIOR
# ============================================================
#
# Business question:
#
# "Does the delivery experience of the first order differ
# between customers who eventually repeat and customers who
# remain one-time buyers?"
#
# If delivery speed is meaningfully associated with repeat
# behavior, it may be useful as a secondary targeting feature.
# ============================================================


# ------------------------------------------------------------
# 1. Load delivery dates from the original orders dataframe
# ------------------------------------------------------------
#
# We only need the order ID and delivery timestamp.
# ------------------------------------------------------------

delivery_dates = orders[
    [
        "order_id",
        "order_delivered_customer_date"
    ]
].copy()


# ------------------------------------------------------------
# 2. Convert delivery timestamp to datetime
# ------------------------------------------------------------

delivery_dates["order_delivered_customer_date"] = pd.to_datetime(
    delivery_dates["order_delivered_customer_date"]
)


# ------------------------------------------------------------
# 3. Attach delivery date to first-order dataset
# ------------------------------------------------------------
#
# Each customer has exactly one first order, so this merge
# remains at one row per customer.
# ------------------------------------------------------------

first_orders_delivery = first_orders.merge(
    delivery_dates,
    on="order_id",
    how="left"
)


# ------------------------------------------------------------
# 4. Calculate delivery time
# ------------------------------------------------------------
#
# Delivery time = 
# customer delivery timestamp - purchase timestamp
#
# The result is expressed in days.
# ------------------------------------------------------------

first_orders_delivery["delivery_days"] = (
    first_orders_delivery["order_delivered_customer_date"]
    - first_orders_delivery["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)


# ------------------------------------------------------------
# 5. Keep only valid delivery durations
# ------------------------------------------------------------
#
# Some orders may not have a delivery date.
# We exclude missing values from the delivery analysis.
#
# We also exclude negative values because they indicate
# inconsistent timestamps rather than a real delivery duration.
# ------------------------------------------------------------

first_orders_delivery_valid = first_orders_delivery[
    first_orders_delivery["delivery_days"].notna()
    & (first_orders_delivery["delivery_days"] >= 0)
].copy()


# ------------------------------------------------------------
# 6. Compare delivery experience by customer type
# ------------------------------------------------------------

delivery_repeat_analysis = (
    first_orders_delivery_valid
    .groupby("customer_type")
    .agg(
        customers=(
            "customer_unique_id",
            "nunique"
        ),

        average_delivery_days=(
            "delivery_days",
            "mean"
        ),

        median_delivery_days=(
            "delivery_days",
            "median"
        ),

        min_delivery_days=(
            "delivery_days",
            "min"
        ),

        max_delivery_days=(
            "delivery_days",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. Round values for readability
# ------------------------------------------------------------

delivery_repeat_analysis[
    [
        "average_delivery_days",
        "median_delivery_days",
        "min_delivery_days",
        "max_delivery_days"
    ]
] = delivery_repeat_analysis[
    [
        "average_delivery_days",
        "median_delivery_days",
        "min_delivery_days",
        "max_delivery_days"
    ]
].round(2)


# ------------------------------------------------------------
# 8. Display the result
# ------------------------------------------------------------

display(delivery_repeat_analysis)

,customer_type,customers,average_delivery_days,median_delivery_days,min_delivery_days,max_delivery_days
0,One-time,90377,12.58,10.21,0.53,209.63
1,Repeat,2877,12.40,10.41,1.04,88.24


In [19]:
# ============================================================
# FIRST-ORDER COHORT VS REPEAT-PURCHASE BEHAVIOR
# ============================================================
#
# Business question:
#
# "Does the time when a customer made their first purchase
# affect whether they eventually became a repeat customer?"
#
# This is important because customers who purchased near the
# end of the dataset have had less time to return.
#
# Therefore, a low repeat rate in a recent cohort may not mean
# that those customers are less loyal. They may simply not have
# had enough observation time.
# ============================================================


# ------------------------------------------------------------
# 1. Create a working copy of the first-order dataset
# ------------------------------------------------------------

cohort_analysis = first_orders[
    [
        "customer_unique_id",
        "order_purchase_timestamp",
        "customer_type"
    ]
].copy()


# ------------------------------------------------------------
# 2. Make sure purchase timestamp is datetime
# ------------------------------------------------------------

cohort_analysis["order_purchase_timestamp"] = pd.to_datetime(
    cohort_analysis["order_purchase_timestamp"]
)


# ------------------------------------------------------------
# 3. Create monthly first-purchase cohorts
# ------------------------------------------------------------
#
# Each customer is assigned to the month in which their
# FIRST order occurred.
#
# Example:
# 2017-10-02 -> 2017-10
# 2018-05-10 -> 2018-05
# ------------------------------------------------------------

cohort_analysis["first_purchase_month"] = (
    cohort_analysis["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)


# ------------------------------------------------------------
# 4. Calculate customers and repeat customers by cohort
# ------------------------------------------------------------

cohort_repeat_analysis = (
    cohort_analysis
    .groupby("first_purchase_month")
    .agg(
        customers=(
            "customer_unique_id",
            "nunique"
        ),

        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Calculate repeat rate
# ------------------------------------------------------------

cohort_repeat_analysis["repeat_rate_pct"] = (
    cohort_repeat_analysis["repeat_customers"]
    / cohort_repeat_analysis["customers"]
    * 100
)


# ------------------------------------------------------------
# 6. Round the repeat rate
# ------------------------------------------------------------

cohort_repeat_analysis["repeat_rate_pct"] = (
    cohort_repeat_analysis["repeat_rate_pct"]
    .round(2)
)


# ------------------------------------------------------------
# 7. Display the cohort analysis
# ------------------------------------------------------------

display(cohort_repeat_analysis)

,first_purchase_month,customers,repeat_customers,repeat_rate_pct
0,2016-09,4,0,0.00
1,2016-10,321,12,3.74
2,2016-12,1,1,100.00
3,2017-01,764,58,7.59
4,2017-02,1752,73,4.17
5,2017-03,2636,130,4.93
6,2017-04,2352,108,4.59
7,2017-05,3596,198,5.51
8,2017-06,3139,173,5.51
9,2017-07,3894,186,4.78


In [21]:
# ============================================================
# CUSTOMER VALUE SEGMENT VS REPEAT BEHAVIOR
# ============================================================
#
# We want to compare:
#   1. Low-value customers
#   2. High-value customers
#
# and determine whether their repeat-purchase rates differ.
#
# We do NOT rely on `customer_features` because that dataframe
# belongs to Notebook 03 and is not available in this kernel.
#
# Instead, we rebuild the required customer-level information
# directly from the analysis dataset.
# ============================================================


# ------------------------------------------------------------
# 1. Create customer-level order statistics
# ------------------------------------------------------------
#
# Each row represents an order.
# We group by customer_unique_id to calculate:
#   - total number of orders
#   - total customer revenue (LTV)
# ------------------------------------------------------------

customer_value = (
    orders_with_values
    .groupby("customer_unique_id")
    .agg(
        total_orders=("order_id", "nunique"),
        total_ltv=("order_value", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. Classify customers as One-time or Repeat
# ------------------------------------------------------------
#
# One-time  -> exactly 1 order
# Repeat    -> more than 1 order
# ------------------------------------------------------------

customer_value["customer_type"] = np.where(
    customer_value["total_orders"] > 1,
    "Repeat",
    "One-time"
)


# ------------------------------------------------------------
# 3. Create value segments
# ------------------------------------------------------------
#
# We use the median LTV as the cutoff, consistent with the
# customer-value segmentation used earlier.
#
# Below median  -> Low-value
# At/above median -> High-value
# ------------------------------------------------------------

ltv_median = customer_value["total_ltv"].median()

customer_value["value_segment"] = np.where(
    customer_value["total_ltv"] >= ltv_median,
    "High-value",
    "Low-value"
)


# ------------------------------------------------------------
# 4. Aggregate repeat behavior by value segment
# ------------------------------------------------------------

value_repeat_summary = (
    customer_value
    .groupby("value_segment")
    .agg(
        customers=("customer_unique_id", "nunique"),

        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),

        average_ltv=("total_ltv", "mean"),

        median_ltv=("total_ltv", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Calculate repeat rate
# ------------------------------------------------------------

value_repeat_summary["repeat_rate_pct"] = (
    value_repeat_summary["repeat_customers"]
    / value_repeat_summary["customers"]
    * 100
)


# ------------------------------------------------------------
# 6. Calculate customer share
# ------------------------------------------------------------

value_repeat_summary["customer_share_pct"] = (
    value_repeat_summary["customers"]
    / value_repeat_summary["customers"].sum()
    * 100
)


# ------------------------------------------------------------
# 7. Round values for readability
# ------------------------------------------------------------

value_repeat_summary[
    [
        "average_ltv",
        "median_ltv",
        "repeat_rate_pct",
        "customer_share_pct"
    ]
] = value_repeat_summary[
    [
        "average_ltv",
        "median_ltv",
        "repeat_rate_pct",
        "customer_share_pct"
    ]
].round(2)


# ------------------------------------------------------------
# 8. Display the result
# ------------------------------------------------------------

print("LTV median cutoff:", round(ltv_median, 2))
print()

display(value_repeat_summary)

NameError: name 'orders_with_values' is not defined

In [23]:
# Check which dataframes are currently available in Notebook 04.
# We are looking specifically for customer, order, payment and
# customer-address data that can be used to rebuild the analysis.

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\n{name}")
        print("Shape:", obj.shape)
        print("Columns:", obj.columns.tolist())


_
Shape: (5, 10)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_state']

__
Shape: (5, 10)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_state']

___
Shape: (5, 3)
Columns: ['customer_id', 'customer_unique_id', 'customer_state']

customers
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

orders
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

order_items
Shape: (112650, 7)
Co

In [24]:
# ============================================================
# CUSTOMER LTV VS REPEAT BEHAVIOR
# ============================================================
#
# Objective:
# Determine whether customers with higher lifetime value (LTV)
# are more likely to become repeat customers.
#
# We use `customer_base`, which was already created from the
# complete 96,096-customer analysis population.
#
# Important:
# We do NOT rebuild the customer-level dataset here because
# customer_base already contains:
#   - total_orders
#   - total_ltv
#   - customer_type
#
# This keeps Notebook 04 consistent with the earlier analysis.
# ============================================================


# ------------------------------------------------------------
# 1. Create a working copy
# ------------------------------------------------------------
#
# Using a copy prevents accidental modification of the
# original customer_base dataframe.

value_repeat_analysis = customer_base[
    [
        "customer_unique_id",
        "total_orders",
        "total_ltv",
        "customer_type"
    ]
].copy()


# ------------------------------------------------------------
# 2. Create LTV value segments
# ------------------------------------------------------------
#
# We use the median LTV as the dividing point.
#
# Customers below the median:
#     Low-value
#
# Customers at or above the median:
#     High-value
#
# This creates two approximately balanced customer groups
# while allowing us to compare repeat behavior.

ltv_cutoff = value_repeat_analysis["total_ltv"].median()

value_repeat_analysis["value_segment"] = np.where(
    value_repeat_analysis["total_ltv"] >= ltv_cutoff,
    "High-value",
    "Low-value"
)


# ------------------------------------------------------------
# 3. Aggregate customer behavior by value segment
# ------------------------------------------------------------
#
# For each segment we calculate:
#
# customers
#     Number of unique customers
#
# repeat_customers
#     Number of customers who placed more than one order
#
# average_ltv
#     Mean lifetime value
#
# median_ltv
#     Median lifetime value

value_repeat_summary = (
    value_repeat_analysis
    .groupby("value_segment")
    .agg(
        customers=("customer_unique_id", "nunique"),

        repeat_customers=(
            "customer_type",
            lambda x: (x == "Repeat").sum()
        ),

        average_ltv=("total_ltv", "mean"),

        median_ltv=("total_ltv", "median")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 4. Calculate repeat rate
# ------------------------------------------------------------
#
# Repeat rate =
#
#     Repeat customers
#     ----------------- × 100
#     Total customers
#
# This allows us to compare repeat behavior between
# low-value and high-value customers.

value_repeat_summary["repeat_rate_pct"] = (
    value_repeat_summary["repeat_customers"]
    / value_repeat_summary["customers"]
    * 100
)


# ------------------------------------------------------------
# 5. Calculate customer share
# ------------------------------------------------------------
#
# This tells us what proportion of the entire customer base
# belongs to each value segment.

value_repeat_summary["customer_share_pct"] = (
    value_repeat_summary["customers"]
    / value_repeat_summary["customers"].sum()
    * 100
)


# ------------------------------------------------------------
# 6. Round numerical results
# ------------------------------------------------------------

value_repeat_summary[
    [
        "average_ltv",
        "median_ltv",
        "repeat_rate_pct",
        "customer_share_pct"
    ]
] = value_repeat_summary[
    [
        "average_ltv",
        "median_ltv",
        "repeat_rate_pct",
        "customer_share_pct"
    ].copy()
].round(2)


# ------------------------------------------------------------
# 7. Sort the segments for easier interpretation
# ------------------------------------------------------------

value_repeat_summary["segment_order"] = (
    value_repeat_summary["value_segment"]
    .map({
        "Low-value": 0,
        "High-value": 1
    })
)

value_repeat_summary = (
    value_repeat_summary
    .sort_values("segment_order")
    .drop(columns="segment_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Display the analysis
# ------------------------------------------------------------

print("LTV cutoff (median):", round(ltv_cutoff, 2))
print()

print("Customer LTV vs Repeat Behavior:")
display(value_repeat_summary)

LTV cutoff (median): 108.0

Customer LTV vs Repeat Behavior:


,value_segment,customers,repeat_customers,average_ltv,median_ltv,repeat_rate_pct,customer_share_pct
0,Low-value,48030,363,63.69,63.11,0.76,49.98
1,High-value,48066,2634,269.41,183.50,5.48,50.02


In [25]:
# ============================================================
# HIGH-VALUE ONE-TIME CUSTOMERS BY FIRST-ORDER VALUE BAND
# ============================================================
#
# Goal:
# Identify one-time customers who have high customer value (LTV)
# and examine how their first-order value is distributed.
#
# These customers are particularly important because:
# - They have already spent relatively more with the business.
# - They have not yet made a repeat purchase.
# - Converting them into repeat customers could therefore
#   generate meaningful incremental revenue.
# ============================================================


# ------------------------------------------------------------
# 1. Select only ONE-TIME customers
# ------------------------------------------------------------
# These customers have made exactly one purchase and therefore
# represent the population that can potentially be converted
# through a repeat-purchase incentive.

one_time_customers = first_orders[
    first_orders["customer_type"] == "One-time"
].copy()


# ------------------------------------------------------------
# 2. Identify HIGH-VALUE one-time customers
# ------------------------------------------------------------
# We previously calculated the median customer LTV as 108.00.
# Customers with LTV >= 108 are classified as high-value.
#
# Since a one-time customer's LTV is essentially their first
# order value, this identifies one-time customers whose
# first purchase was relatively valuable.

ltv_cutoff = customer_base["total_ltv"].median()

high_value_one_time = one_time_customers[
    one_time_customers["total_ltv"] >= ltv_cutoff
].copy()


# ------------------------------------------------------------
# 3. Create first-order value bands
# ------------------------------------------------------------
# This allows us to see where high-value one-time customers
# are concentrated in terms of their initial purchase value.

high_value_one_time["first_order_value_band"] = pd.cut(
    high_value_one_time["order_value"],
    bins=[-float("inf"), 50, 100, 250, 500, 1000, float("inf")],
    labels=[
        "< $50",
        "$50–$99",
        "$100–$249",
        "$250–$499",
        "$500–$999",
        "$1,000+"
    ],
    right=False
)


# ------------------------------------------------------------
# 4. Aggregate the results by first-order value band
# ------------------------------------------------------------
# We calculate:
# - number of high-value one-time customers
# - average first-order value
# - median first-order value
# - total revenue represented by each segment

high_value_one_time_analysis = (
    high_value_one_time
    .groupby("first_order_value_band", observed=False)
    .agg(
        customers=("customer_unique_id", "nunique"),
        average_first_order_value=("order_value", "mean"),
        median_first_order_value=("order_value", "median"),
        total_revenue=("order_value", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Calculate customer and revenue shares
# ------------------------------------------------------------
# Customer share tells us how large each segment is.
# Revenue share tells us how much of the high-value one-time
# customer revenue comes from each segment.

total_high_value_customers = high_value_one_time_analysis[
    "customers"
].sum()

total_high_value_revenue = high_value_one_time_analysis[
    "total_revenue"
].sum()

high_value_one_time_analysis["customer_share_pct"] = (
    high_value_one_time_analysis["customers"]
    / total_high_value_customers
    * 100
)

high_value_one_time_analysis["revenue_share_pct"] = (
    high_value_one_time_analysis["total_revenue"]
    / total_high_value_revenue
    * 100
)


# ------------------------------------------------------------
# 6. Round the numerical columns
# ------------------------------------------------------------
# This makes the final output easier to read and use in the
# project report.

high_value_one_time_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "total_revenue",
        "customer_share_pct",
        "revenue_share_pct"
    ]
] = high_value_one_time_analysis[
    [
        "average_first_order_value",
        "median_first_order_value",
        "total_revenue",
        "customer_share_pct",
        "revenue_share_pct"
    ]
].round(2)


# ------------------------------------------------------------
# 7. Display the final analysis
# ------------------------------------------------------------

print("LTV cutoff:", round(ltv_cutoff, 2))
print("High-value one-time customers:", len(high_value_one_time))

high_value_one_time_analysis

LTV cutoff: 108.0
High-value one-time customers: 45432


,first_order_value_band,customers,average_first_order_value,median_first_order_value,total_revenue,customer_share_pct,revenue_share_pct
0,< $50,0,NaN,NaN,0.00,0.00,0.00
1,$50–$99,0,NaN,NaN,0.00,0.00,0.00
2,$100–$249,32516,161.38,154.84,"5,247,325.76",71.57,43.60
3,$250–$499,8825,337.31,322.95,"2,976,761.52",19.42,24.73
4,$500–$999,2969,681.12,653.21,"2,022,231.78",6.54,16.80
5,"$1,000+",1122,"1,594.68","1,367.62","1,789,227.47",2.47,14.87


In [26]:
# ============================================================
# INCENTIVE TARGETING: CONVERSION RATE & ECONOMIC IMPACT
# ============================================================
#
# Objective:
# Estimate the potential incremental revenue and ROI from
# targeting high-value one-time customers with a repeat-
# purchase incentive.
#
# Target population:
#     High-value one-time customers
#
# We evaluate several hypothetical conversion rates and
# incentive costs per successful conversion.
#
# IMPORTANT:
# These are scenario estimates, NOT observed conversions.
# They help determine whether an incentive could be
# economically viable.
# ============================================================


# ------------------------------------------------------------
# 1. Define the target population
# ------------------------------------------------------------
#
# We already identified 45,432 high-value one-time customers.
# These customers are the target population for the incentive.

target_customers = len(high_value_one_time)

print("Target customers:", target_customers)


# ------------------------------------------------------------
# 2. Estimate the average second-order value
# ------------------------------------------------------------
#
# We use the observed second-order value from customers who
# actually became repeat purchasers.
#
# `orders_sorted` contains purchase sequence information.
# Purchase number = 2 identifies second purchases.
# ------------------------------------------------------------

second_orders = orders_sorted[
    orders_sorted["purchase_number"] == 2
].copy()


# Calculate the average value of an observed second purchase.

average_second_order_value = second_orders["order_value"].mean()

print(
    "Average observed second-order value:",
    round(average_second_order_value, 2)
)


# ------------------------------------------------------------
# 3. Define hypothetical conversion-rate scenarios
# ------------------------------------------------------------
#
# Example:
# 1% conversion means 1% of the 45,432 target customers
# are assumed to make a repeat purchase because of the
# incentive.
#
# These are scenario assumptions rather than actual results.

conversion_rates = [1, 3, 5, 10]


# ------------------------------------------------------------
# 4. Define incentive costs
# ------------------------------------------------------------
#
# Cost is assumed to be incurred only when a customer
# successfully converts.
#
# This is a conservative and simple economic model.

incentive_costs = [5, 10, 15, 25, 50, 75, 100]


# ------------------------------------------------------------
# 5. Generate all conversion-rate × incentive-cost scenarios
# ------------------------------------------------------------

scenarios = []


for conversion_rate in conversion_rates:

    # Estimate number of customers who convert.

    estimated_conversions = round(
        target_customers * conversion_rate / 100
    )


    # Estimate gross incremental revenue generated by
    # those additional second purchases.

    estimated_revenue = (
        estimated_conversions
        * average_second_order_value
    )


    # Evaluate different incentive costs.

    for incentive_cost in incentive_costs:

        # Total incentive expenditure.

        total_incentive_cost = (
            estimated_conversions
            * incentive_cost
        )


        # Revenue remaining after incentive expense.

        net_incremental_revenue = (
            estimated_revenue
            - total_incentive_cost
        )


        # ROI:
        #
        # Net incremental revenue
        # ---------------------- × 100
        # Incentive cost

        if total_incentive_cost > 0:

            roi_pct = (
                net_incremental_revenue
                / total_incentive_cost
                * 100
            )

        else:

            roi_pct = np.nan


        scenarios.append(
            {
                "conversion_rate_pct": conversion_rate,
                "estimated_conversions": estimated_conversions,
                "average_second_order_value":
                    average_second_order_value,
                "estimated_revenue":
                    estimated_revenue,
                "incentive_cost_per_conversion":
                    incentive_cost,
                "total_incentive_cost":
                    total_incentive_cost,
                "net_incremental_revenue":
                    net_incremental_revenue,
                "roi_pct":
                    roi_pct
            }
        )


# ------------------------------------------------------------
# 6. Convert scenarios into a dataframe
# ------------------------------------------------------------

incentive_scenarios = pd.DataFrame(scenarios)


# ------------------------------------------------------------
# 7. Round monetary and percentage values
# ------------------------------------------------------------

incentive_scenarios[
    [
        "average_second_order_value",
        "estimated_revenue",
        "incentive_cost_per_conversion",
        "total_incentive_cost",
        "net_incremental_revenue",
        "roi_pct"
    ]
] = incentive_scenarios[
    [
        "average_second_order_value",
        "estimated_revenue",
        "incentive_cost_per_conversion",
        "total_incentive_cost",
        "net_incremental_revenue",
        "roi_pct"
    ]
].round(2)


# ------------------------------------------------------------
# 8. Display the complete scenario table
# ------------------------------------------------------------

print("\nIncentive Economics Scenarios:")

display(incentive_scenarios)

Target customers: 45432
Average observed second-order value: 148.51

Incentive Economics Scenarios:


,conversion_rate_pct,estimated_conversions,average_second_order_value,estimated_revenue,incentive_cost_per_conversion,total_incentive_cost,net_incremental_revenue,roi_pct
0,1,454,148.51,"67,425.30",5,2270,"65,155.30","2,870.28"
1,1,454,148.51,"67,425.30",10,4540,"62,885.30","1,385.14"
2,1,454,148.51,"67,425.30",15,6810,"60,615.30",890.09
3,1,454,148.51,"67,425.30",25,11350,"56,075.30",494.06
4,1,454,148.51,"67,425.30",50,22700,"44,725.30",197.03
5,1,454,148.51,"67,425.30",75,34050,"33,375.30",98.02
6,1,454,148.51,"67,425.30",100,45400,"22,025.30",48.51
7,3,1363,148.51,"202,424.41",5,6815,"195,609.41","2,870.28"
8,3,1363,148.51,"202,424.41",10,13630,"188,794.41","1,385.14"
9,3,1363,148.51,"202,424.41",15,20445,"181,979.41",890.09


In [27]:
# ============================================================
# FINAL CUSTOMER TARGETING RECOMMENDATION
# ============================================================
#
# Goal:
# Combine the major findings from Notebook 04 into a single
# decision-oriented table.
#
# This table does NOT create new assumptions from the raw data.
# Instead, it summarizes the evidence already calculated:
#
# 1. High-value customers have much higher repeat rates.
# 2. High-value one-time customers form a large target pool.
# 3. Higher first-order value bands contribute disproportionately
#    more revenue.
# 4. A $25 incentive remains economically attractive under the
#    scenario model.
# ============================================================


# ------------------------------------------------------------
# 1. Key metrics from the completed analysis
# ------------------------------------------------------------

target_customers = len(high_value_one_time)

target_average_ltv = high_value_one_time["total_ltv"].mean()

target_median_ltv = high_value_one_time["total_ltv"].median()

target_revenue = high_value_one_time["total_ltv"].sum()

target_customer_share = (
    target_customers
    / len(customer_base)
    * 100
)


# ------------------------------------------------------------
# 2. Calculate the target segment's share of total customer
#    revenue.
# ------------------------------------------------------------

total_customer_revenue = customer_base["total_ltv"].sum()

target_revenue_share = (
    target_revenue
    / total_customer_revenue
    * 100
)


# ------------------------------------------------------------
# 3. Historical repeat-rate comparison
# ------------------------------------------------------------
#
# These values come from the LTV segmentation analysis.

low_value_repeat_rate = 0.76
high_value_repeat_rate = 5.48

repeat_rate_lift = (
    high_value_repeat_rate
    / low_value_repeat_rate
)


# ------------------------------------------------------------
# 4. Select a reference incentive scenario
# ------------------------------------------------------------
#
# We use:
#   - 5% hypothetical conversion
#   - $25 incentive per successful conversion
#
# This is NOT an observed conversion rate.
# It is simply a practical scenario for illustrating the
# potential economics.

reference_conversion_rate = 5

reference_incentive = 25

reference_conversions = round(
    target_customers
    * reference_conversion_rate
    / 100
)

reference_revenue = (
    reference_conversions
    * average_second_order_value
)

reference_incentive_cost = (
    reference_conversions
    * reference_incentive
)

reference_net_revenue = (
    reference_revenue
    - reference_incentive_cost
)

reference_roi = (
    reference_net_revenue
    / reference_incentive_cost
    * 100
)


# ------------------------------------------------------------
# 5. Build the final decision table
# ------------------------------------------------------------

final_decision = pd.DataFrame(
    {
        "metric": [
            "Target segment",
            "Target customers",
            "Target average LTV",
            "Target median LTV",
            "Target customer share (%)",
            "Target revenue share (%)",
            "High-value repeat rate (%)",
            "Low-value repeat rate (%)",
            "Repeat-rate multiple",
            "Average second-order value",
            "Reference conversion scenario (%)",
            "Estimated conversions",
            "Reference incentive per conversion",
            "Estimated incremental revenue",
            "Estimated incentive cost",
            "Estimated net incremental revenue",
            "Estimated ROI (%)"
        ],

        "value": [
            "High-value one-time customers",
            target_customers,
            round(target_average_ltv, 2),
            round(target_median_ltv, 2),
            round(target_customer_share, 2),
            round(target_revenue_share, 2),
            high_value_repeat_rate,
            low_value_repeat_rate,
            round(repeat_rate_lift, 2),
            round(average_second_order_value, 2),
            reference_conversion_rate,
            reference_conversions,
            reference_incentive,
            round(reference_revenue, 2),
            round(reference_incentive_cost, 2),
            round(reference_net_revenue, 2),
            round(reference_roi, 2)
        ]
    }
)


# ------------------------------------------------------------
# 6. Display the final decision table
# ------------------------------------------------------------

print("FINAL TARGETING DECISION")
print("=" * 60)

display(final_decision)

FINAL TARGETING DECISION


,metric,value
0,Target segment,High-value one-time customers
1,Target customers,45432
2,Target average LTV,264.91
3,Target median LTV,180.21
4,Target customer share (%),47.28
5,Target revenue share (%),75.18
6,High-value repeat rate (%),5.48
7,Low-value repeat rate (%),0.76
8,Repeat-rate multiple,7.21
9,Average second-order value,148.51
